# TRNRun examples

In [ ]:
import shutil
from pathlib import Path

from trnrun import SimulationConfig, SimulationManager

__root__ = Path.cwd()

TRNEXE_PATH = Path(r"C:\TRNSYS18\Exe\TrnEXE64.exe")
MASTER_DCK = __root__ / "dck" / "example_wo_plot_w_tracking.dck"
DCK_FOLDER = __root__ / "runs"

CONFIG = SimulationConfig(trnexe_path=TRNEXE_PATH, watch_tmp=True)
REFRESH_INTERVAL = 0.1

## Single simulation

In [ ]:
with SimulationManager(max_concurrent=1, refresh_interval=REFRESH_INTERVAL) as manager:
    simulation = manager.add(MASTER_DCK, CONFIG)
    manager.wait(simulation)

simulation.status

## Multiple simulations

In [ ]:
SIM_COUNT = 100
MAX_CONCURRENT = 50

DCK_FOLDER.mkdir(parents=True, exist_ok=True)

dck_files = []
for i in range(1, SIM_COUNT + 1):
    dck = DCK_FOLDER / f"{MASTER_DCK.stem}_{i:03d}{MASTER_DCK.suffix}"
    shutil.copyfile(MASTER_DCK, dck)
    dck_files.append(dck)

dck_files[:5]

In [ ]:
with SimulationManager(
    max_concurrent=MAX_CONCURRENT,
    refresh_interval=REFRESH_INTERVAL,
) as manager:
    for dck in dck_files:
        manager.add(dck, CONFIG)
    manager.wait()

{
    "total": len(manager.simulations),
    "succeeded": len(manager.succeeded),
    "failed": len(manager.failed),
}